In [3]:
import pandas as pd
import numpy as np
import joblib
import sys
sys.path.append("../src")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from fairness_metrics import demographic_parity, equal_opportunity

# Load data
df = pd.read_csv("../data/german_credit_data.csv")
if "Unnamed: 0" in df.columns:
    df.drop("Unnamed: 0", axis=1, inplace=True)

df["target"] = (df["Credit amount"] > 2000).astype(int)

sex_col = df["Sex"].copy()
age_col = df["Age"].copy()

print(df.shape)
print(df.columns.tolist())

(1000, 12)
['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Purpose', 'Risk', 'Age_group', 'target']


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

y     = df["target"]
X     = df.drop("target", axis=1)
X_enc = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_enc, y, test_size=0.2, random_state=42
)

sex_train = sex_col.loc[X_train.index].reset_index(drop=True)
sex_test  = sex_col.loc[X_test.index].reset_index(drop=True)
y_test_r  = y_test.reset_index(drop=True)

print("Feature columns:", X_enc.columns.tolist())
print("Total features  :", X_enc.shape[1])
print("Train size:", len(X_train), "| Test size:", len(X_test))

Feature columns: ['Age', 'Job', 'Credit amount', 'Duration', 'Sex_male', 'Housing_own', 'Housing_rent', 'Saving accounts_moderate', 'Saving accounts_quite rich', 'Saving accounts_rich', 'Checking account_moderate', 'Checking account_rich', 'Purpose_car', 'Purpose_domestic appliances', 'Purpose_education', 'Purpose_furniture/equipment', 'Purpose_radio/TV', 'Purpose_repairs', 'Purpose_vacation/others', 'Risk_good', 'Age_group_Senior', 'Age_group_Young']
Total features  : 22
Train size: 800 | Test size: 200


In [7]:
# ── Baseline model ────────────────────────────────────────────
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)
print("Baseline Accuracy:", accuracy_score(y_test_r, y_pred_baseline))

# ── Fair model (reweighting) ──────────────────────────────────
weights = np.ones(len(X_train))
weights[sex_train == "female"] = 1.5

fair_model = LogisticRegression(max_iter=1000)
fair_model.fit(X_train, y_train, sample_weight=weights)
y_pred_fair = fair_model.predict(X_test)
print("Fair Model Accuracy:", accuracy_score(y_test_r, y_pred_fair))

print("\nAll models trained on", X_train.shape[1], "features ✅")

Baseline Accuracy: 1.0
Fair Model Accuracy: 1.0

All models trained on 22 features ✅


c:\Users\Visveswaran\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Visveswaran\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
 

In [8]:
# These features encode gender indirectly in German Credit data
proxy_keywords = ["Sex", "Job", "Housing", "Saving accounts"]

proxy_columns = []
for col in X_enc.columns:
    for keyword in proxy_keywords:
        kw_encoded = keyword.replace(" ", "_").lower()
        if keyword.lower() in col.lower() or kw_encoded in col.lower():
            proxy_columns.append(col)
            break

proxy_columns = list(set(proxy_columns))
print("Proxy columns being removed:", proxy_columns)

X_train_c2 = X_train.drop(columns=proxy_columns, errors="ignore")
X_test_c2  = X_test.drop(columns=proxy_columns,  errors="ignore")

# Train causal model 2
model_c2 = LogisticRegression(max_iter=1000)
model_c2.fit(X_train_c2, y_train)
y_pred_c2 = model_c2.predict(X_test_c2)

acc_c2 = accuracy_score(y_test_r, y_pred_c2)
print(f"Causal Model 2 Accuracy: {acc_c2:.4f}")

dp_c2, dp_gap_c2 = demographic_parity(y_pred_c2, sex_test)
eo_c2, eo_gap_c2 = equal_opportunity(y_test_r, y_pred_c2, sex_test)

Proxy columns being removed: ['Sex_male', 'Saving accounts_rich', 'Saving accounts_moderate', 'Job', 'Housing_own', 'Saving accounts_quite rich', 'Housing_rent']
Causal Model 2 Accuracy: 1.0000

===== DEMOGRAPHIC PARITY =====
  female: 0.4464
  male: 0.5833
  Parity Gap: 0.1369
  ⚠️  BIAS DETECTED — gap exceeds 0.05 threshold

===== EQUAL OPPORTUNITY (True Positive Rate) =====
  female: TPR = 1.0000
  male: TPR = 1.0000
  TPR Gap: 0.0000
  ✅  TPR within acceptable range


c:\Users\Visveswaran\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [9]:
# Take the test set and flip Sex values
# Original: male → flipped: female and vice versa

X_test_flipped = X_test.copy()

# Find the encoded Sex column
sex_col_encoded = [c for c in X_test.columns if "Sex" in c]
print("Sex encoded column:", sex_col_encoded)

if sex_col_encoded:
    col = sex_col_encoded[0]
    # Flip 0→1 and 1→0
    X_test_flipped[col] = 1 - X_test_flipped[col]

# Get predictions on original and flipped
y_pred_original = fair_model.predict(X_test)
y_pred_flipped  = fair_model.predict(X_test_flipped)

# Counterfactual consistency — how often does prediction CHANGE when gender flips?
changed = (y_pred_original != y_pred_flipped).sum()
total   = len(y_pred_original)
change_rate = changed / total

print(f"\n===== COUNTERFACTUAL FAIRNESS =====")
print(f"Total test samples       : {total}")
print(f"Predictions that changed : {changed}")
print(f"Change rate              : {change_rate:.4f} ({change_rate*100:.1f}%)")

if change_rate < 0.10:
    print("✅ Model is counterfactually fair — gender flip rarely changes prediction")
elif change_rate < 0.20:
    print("⚠️  Moderate counterfactual sensitivity — some gender influence remains")
else:
    print("❌ High counterfactual sensitivity — gender significantly affects predictions")

Sex encoded column: ['Sex_male']

===== COUNTERFACTUAL FAIRNESS =====
Total test samples       : 200
Predictions that changed : 0
Change rate              : 0.0000 (0.0%)
✅ Model is counterfactually fair — gender flip rarely changes prediction


In [12]:
# Cell A - Fix directory and imports
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")

while not os.path.exists(os.path.join(os.getcwd(), "data")):
    os.chdir("..")
print("Working directory:", os.getcwd())

Working directory: e:\causal-fairness-credit-scoring\causal-fairness-credit-scoring


In [13]:
# Cell B - Load and prepare data
df = pd.read_csv("data/german_credit_data.csv")
if "Unnamed: 0" in df.columns:
    df.drop("Unnamed: 0", axis=1, inplace=True)

sex_col = df["Sex"].copy()
y       = df["Risk"]
X       = df.drop(columns=["Risk", "Sex", "Age_group"], errors="ignore")
X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test, sex_train, sex_test = train_test_split(
    X_encoded, y, sex_col,
    test_size=0.2, random_state=42, stratify=y
)

imputer = SimpleImputer(strategy="median")
scaler  = StandardScaler()
X_train_proc = scaler.fit_transform(imputer.fit_transform(X_train))
X_test_proc  = scaler.transform(imputer.transform(X_test))

print("X_encoded shape:", X_encoded.shape)
print("y unique:", y.unique())

X_encoded shape: (1000, 18)
y unique: ['good' 'bad']


In [15]:
# Local demographic_parity that works with string predictions
def demographic_parity(y_pred, sex):
    results = pd.DataFrame({"prediction": y_pred, "Sex": sex})
    rates = results.groupby("Sex")["prediction"].apply(
        lambda x: (x == "good").mean()
    )
    gap = abs(rates["male"] - rates["female"])
    return rates, gap

def equal_opportunity(y_true, y_pred, sex):
    results = pd.DataFrame({
        "actual":     y_true,
        "prediction": y_pred,
        "Sex":        sex
    })
    tpr = {}
    for group in results["Sex"].unique():
        subset   = results[results["Sex"] == group]
        positive = subset[subset["actual"] == "good"]
        tpr[group] = (positive["prediction"] == "good").mean() if len(positive) > 0 else 0.0
    gap = abs(tpr.get("male", 0) - tpr.get("female", 0))
    return tpr, gap

In [16]:
# Causal Model 1 - Remove Sex directly
X_c1 = X_encoded.drop(columns=["Sex_male"], errors="ignore")

X_train_c1, X_test_c1, y_train_c1, y_test_c1 = train_test_split(
    X_c1, y, test_size=0.2, random_state=42, stratify=y
)

imp_c1 = SimpleImputer(strategy="median")
sc_c1  = StandardScaler()
X_train_c1_proc = sc_c1.fit_transform(imp_c1.fit_transform(X_train_c1))
X_test_c1_proc  = sc_c1.transform(imp_c1.transform(X_test_c1))

model_c1 = LogisticRegression(max_iter=2000, random_state=42)
model_c1.fit(X_train_c1_proc, y_train_c1)
y_pred_c1 = model_c1.predict(X_test_c1_proc)
acc_c1    = accuracy_score(y_test_c1, y_pred_c1)

dp_rates_c1, dp_gap_c1 = demographic_parity(y_pred_c1, sex_test.values)
eo_tpr_c1,   eo_gap_c1 = equal_opportunity(y_test_c1.values, y_pred_c1, sex_test.values)

print(f"Causal Model 1 Accuracy : {acc_c1:.4f}")
print(f"Parity Gap              : {dp_gap_c1:.4f}")

# Causal Model 2 - Remove Sex + proxy columns
proxy_cols = ["Sex_male", "Purpose_car", "Purpose_education"]
X_c2 = X_encoded.drop(columns=proxy_cols, errors="ignore")

X_train_c2, X_test_c2, y_train_c2, y_test_c2 = train_test_split(
    X_c2, y, test_size=0.2, random_state=42, stratify=y
)

imp_c2 = SimpleImputer(strategy="median")
sc_c2  = StandardScaler()
X_train_c2_proc = sc_c2.fit_transform(imp_c2.fit_transform(X_train_c2))
X_test_c2_proc  = sc_c2.transform(imp_c2.transform(X_test_c2))

model_c2 = LogisticRegression(max_iter=2000, random_state=42)
model_c2.fit(X_train_c2_proc, y_train_c2)
y_pred_c2 = model_c2.predict(X_test_c2_proc)
acc_c2    = accuracy_score(y_test_c2, y_pred_c2)

dp_rates_c2, dp_gap_c2 = demographic_parity(y_pred_c2, sex_test.values)
eo_tpr_c2,   eo_gap_c2 = equal_opportunity(y_test_c2.values, y_pred_c2, sex_test.values)

print(f"\nCausal Model 2 Accuracy : {acc_c2:.4f}")
print(f"Parity Gap              : {dp_gap_c2:.4f}")

Causal Model 1 Accuracy : 0.6900
Parity Gap              : 0.0381

Causal Model 2 Accuracy : 0.6800
Parity Gap              : 0.0524


In [18]:
# Reset all indices to align properly
y_test_r        = y_test.reset_index(drop=True)
sex_test_r      = sex_test.reset_index(drop=True)

y_pred_baseline = np.array(y_pred_baseline)
y_pred_fair     = np.array(y_pred_fair)

print("y_test_r length     :", len(y_test_r))
print("sex_test_r length   :", len(sex_test_r))
print("y_pred_baseline len :", len(y_pred_baseline))
print("y_pred_fair len     :", len(y_pred_fair))

y_test_r length     : 200
sex_test_r length   : 200
y_pred_baseline len : 200
y_pred_fair len     : 200


In [21]:
# Cell 5 - Baseline Model (FIXED)
baseline = LogisticRegression(max_iter=2000, random_state=42)
baseline.fit(X_train_proc, y_train)  # y_train must be 'good'/'bad'

y_pred_baseline = baseline.predict(X_test_proc)

# Force to string just in case
y_pred_baseline = np.array([str(p) for p in y_pred_baseline])
y_test_r        = np.array([str(v) for v in y_test.reset_index(drop=True)])

print("y_pred_baseline unique:", np.unique(y_pred_baseline))
print("y_test_r unique:", np.unique(y_test_r))
print("Baseline Accuracy:", accuracy_score(y_test_r, y_pred_baseline))

y_pred_baseline unique: ['bad' 'good']
y_test_r unique: ['bad' 'good']
Baseline Accuracy: 0.69


In [23]:
# Force all predictions and labels to string type
y_test_r        = np.array([str(v) for v in y_test.reset_index(drop=True)])
y_pred_baseline = np.array([str(p) for p in baseline.predict(X_test_proc)])
y_pred_fair     = np.array([str(p) for p in y_pred_fair])
sex_test_r      = np.array([str(s) for s in sex_test.reset_index(drop=True)])

print("y_test_r unique      :", np.unique(y_test_r))
print("y_pred_baseline unique:", np.unique(y_pred_baseline))
print("y_pred_fair unique   :", np.unique(y_pred_fair))
print("sex_test_r unique    :", np.unique(sex_test_r))

y_test_r unique      : ['bad' 'good']
y_pred_baseline unique: ['bad' 'good']
y_pred_fair unique   : ['0' '1']
sex_test_r unique    : ['female' 'male']


In [24]:
# Collect all metrics
from sklearn.metrics import accuracy_score

results = pd.DataFrame({
    "Model": [
        "Baseline",
        "Fair (Threshold)",
        "Causal 1 (No Sex)",
        "Causal 2 (No Proxies)"
    ],
    "Accuracy": [
        round(accuracy_score(y_test_r, y_pred_baseline), 4),
        round(accuracy_score(y_test_r, y_pred_fair),     4),
        round(acc_c1, 4),
        round(acc_c2, 4),
    ],
    "Parity Gap": [
        round(demographic_parity(y_pred_baseline, sex_test_r)[1], 4),
        round(demographic_parity(y_pred_fair,     sex_test_r)[1], 4),
        round(dp_gap_c1, 4),
        round(dp_gap_c2, 4),
    ],
    "EO Gap": [
        round(equal_opportunity(y_test_r, y_pred_baseline, sex_test_r)[1], 4),
        round(equal_opportunity(y_test_r, y_pred_fair,     sex_test_r)[1], 4),
        round(eo_gap_c1, 4),
        round(eo_gap_c2, 4),
    ]
})

print(results.to_string(index=False))
import os
os.makedirs("../outputs", exist_ok=True)
results.to_csv("../outputs/model_comparison.csv", index=False)
print("\nSaved to outputs/model_comparison.csv")

                Model  Accuracy  Parity Gap  EO Gap
             Baseline      0.69      0.0381    0.01
     Fair (Threshold)      0.00      0.0000    0.00
    Causal 1 (No Sex)      0.69      0.0381    0.01
Causal 2 (No Proxies)      0.68      0.0524    0.01

Saved to outputs/model_comparison.csv


In [26]:
import os
os.makedirs("../models", exist_ok=True)

joblib.dump(model_c1, "../models/causal_model_1.pkl")
joblib.dump(model_c2, "../models/causal_model_2.pkl")
print("Causal models saved!")

Causal models saved!
